## Model Training On Synthetic Data

In [ ]:
# Import modules
import pandas as pd
import numpy as np
import os 
import sys

In [ ]:
# Get absolute path to project root
project_root = os.path.abspath("..")

# Add to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

### Load Dataset

In [ ]:
transactions_data_train = pd.read_csv("data/train.csv")

print(transactions_data_train.head())

## Data Preparation 

### One Hot Encoding

Before trainng any machine learning model, the data has to be in the proper format for the task at hand. Machine learning models work only with numerical data and don't now what to do with text data. Here the merchant category feature is encoded using One Hot Encoding.

In [ ]:
transactions_data_train = pd.get_dummies(transactions_data_train, columns=["merchant_category"], dtype="int")
print(transactions_data_train.head())

In [ ]:
# Predictor features
X = transactions_data_train.drop("is_fraud", axis=1)

# Target labels - fraud, not fraud
y = transactions_data_train["is_fraud"]

### Data Resampling

To slove the class imbalance problem in the dataset the data has to be resampled. Here the minority class of the target variable is over-sampled using SMOTE (Synthetic Minority Over-sampling Technique). This thechinique creates synthetic samples from the minority class thus increasing the number of fraudulent transactions in the dataset.

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(sampling_strategy="minority", random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

print("Target labels distribution before resampling")
print(y.value_counts())

print("Target labels distribution after resampling")
print(y_resampled.value_counts())

As it can be seen on the charts bellow SMOTE increased the number of fraud samples in the dataset and now they are equal to the non-fraud samples

In [ ]:
import matplotlib.pyplot as plt
from src.utils import (
    plot_label_distribution, 
    cross_validate_pipeline, 
    tune_pipeline_hyper_parameters,
    evaluate_pipeline,
    split_data, 
    tune_classification_threshold,
    plot_confusion_matrix_from_pipeline,
    plot_roc_curve_from_pipeline
)

plot_label_distribution(y, "Target Labels Distribution Before Resampling")

In [ ]:
plot_label_distribution(y_resampled, "Target Labels Distribution After Resampling")

## Model Building

It's time to try out a few models and see how they perform on the task. Let's start simple and increase model complexity as we go. Bellow a few popular machine learning classification algorithms are imported 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

### Evaluation Metrics

Given the imbalnced dataset, some suitable metrics are __ROC AUC__ score, __average precision__ score and __f1__ score.

- **ROC AUC**: is the area under the Receiver Operating Characteristic curve

- **average precision**: summarizes a precision-recall curve as the weighted mean of precisions achieved at each threshold, with the increase in recall from the previous threshold used as the weight:

$$ AP = \sum(R_n - R_{n - 1})P_n $$

  where $P_n$ and $ R_n $ are the precision and recall at the nth threshold 

- **f1 score**: is the harmonic mean of precision and recall
$$ F1 = \frac{2 * TP}{2 * TP + FP + FN} $$

### Logistic Regression

Bellow a pipeline was created with StandardScaler, SMOTE oversampling and a LogisticRgression model. The model was trained using stratified cross-validation and achived a roc_aus score of 92%, average precision score of 46% and f1 score of 21%

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline

# Define components
smote = SMOTE(random_state=42)
scaler = StandardScaler() 
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Make pipeline
log_reg_pipeline = ImbPipeline(
    steps= [
        ("scaler", scaler),
        ("smote", smote),
        ("model", log_reg)
    ]
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1"
}

cross_validate_pipeline(log_reg_pipeline, X, y, cv, scoring)

### K-Nearest Neighbors Classifier

Here the KNeighborsClassifier pipeline acived a roc_auc score of 84%, pr_auc score of 23%, and f1 score of 31%

In [ ]:
knn_clf = KNeighborsClassifier()

knn_clf_pipeline = ImbPipeline(
    steps = [
        ("scaler", scaler),
        ("smote", smote),
        ("model", knn_clf)
    ]
)

cross_validate_pipeline(knn_clf_pipeline, X, y, cv, scoring)

### Linear Support Vector Classifier

The pipeline performed the same as the LogisticRegression pipeline

In [ ]:
linear_svc = LinearSVC(random_state=42)

linear_svc_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", linear_svc)
    ]
)

cross_validate_pipeline(linear_svc_pipeline, X, y, cv, scoring)

### Random Forest Classifier

The Random Forest Classifier pipeline achived a roc_auc score of 92%, an average precision score of 34% and a f1 score of 38%. It has similar performance in terms of roc_auc score and f1 score to the Linear SVC pipleine but it has lower average precision score. It also performs better than the KNN pipeline 

In [ ]:
forest_clf = RandomForestClassifier(random_state=42)

forest_clf_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", forest_clf)
    ]
)

cross_validate_pipeline(forest_clf_pipeline, X, y, cv, scoring)

### Gradient Boosting Classifier

The Gradient Boosting Classifier achived a roc auc score of 94%, an average precision score of 48% and a f1 score of 40%. Even with default hyper-parameters this is the best performing model for now. 

In [ ]:
gb_clf = GradientBoostingClassifier(random_state=42)

gb_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", gb_clf)
    ]
)

cross_validate_pipeline(gb_pipeline, X, y, cv, scoring)

## Hyperparameter Tuning

Let's try squeezing more performance form the models by tuning some of ther hyperparameters
by using random searching with cross-validation. Since this method picks random values for each hyperparameter from a predefined grid of values it doesn't guarnatee the optimal values but it's much faster than other alternatives.

### Logistic Regression

After hyper-parmter tuning the best model achived a average precision score of 50% which is slightly better than the model with default hyper-parameters.

In [ ]:
param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__C": [0.001, 0.01, 0.1, 1, 10],
    "model__fit_intercept": [True, False],
    "model__class_weight": [None, "balanced"],
}

best_model, best_params, best_score = tune_pipeline_hyper_parameters(
    log_reg_pipeline, 
    param_dist, 
    20, 
    cv, 
    "average_precision", 
    X, 
    y
)

In [ ]:
print(f"Best hyperparameters: {best_params}")
print(f"Best score: {best_score}")

### K-Nearest Neighbors Classifier

After hyperparameter tuning the best model achived a average_precision score of 46% which is slightly better than the model with default hyper-parameters but it's worse than the Logistic Regression model.

In [ ]:
param_dist = {
    "model__n_neighbors": [20, 22, 24, 26, 28, 30],
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
}

best_model, best_params, best_score = tune_pipeline_hyper_parameters(
    knn_clf_pipeline, 
    param_dist, 
    20, 
    cv, 
    "average_precision", 
    X, y
)

In [ ]:
print(f"Best hyperparameters: {best_params}")
print(f"Best score: {best_score}")

### Linear Support Vector Classifier

 After hyperparameter tuning the best model achived a average precision score of 49% which is slightly better than the model with default hyper-parameters but it's still worse than the Logistic Regression model.

In [ ]:
param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 20, 30, 40],
    "model__loss": ["hinge", "squared_hinge"],
    "model__fit_intercept": [True, False],
    "model__class_weight": [None, "balanced"]
}

best_model, best_params, best_score = tune_pipeline_hyper_parameters(
    linear_svc_pipeline, 
    param_dist, 
    20, 
    cv, 
    "average_precision", 
    X, y
)

In [ ]:
print(f"Best parameters: {best_params}")
print(f"Best score: {best_score}")

### Random Forest Classifer

Bellow the `n_estimators`, `max_depth`, `criterion`, `min_samples_leaf`, `min_samples_split`, `max_features` and `class_weight` hyperparameters are tuned. After hyperparameter tuning the best model achived an average precision score of 53%.

In [ ]:
param_dist = {
    "smote__k_neighbors":[3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__n_estimators": [50, 70, 90],
    "model__max_depth": [2, 4, 6],
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__min_samples_leaf": [2, 4, 6],
    "model__min_samples_split": [2, 4, 6],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

best_model, best_params, best_score = tune_pipeline_hyper_parameters(
    forest_clf_pipeline, 
    param_dist, 
    20, 
    cv, 
    "average_precision", 
    X, y
)

In [ ]:
print(f"Best paramters: {best_params}")
print(f"Best score: {best_score}")

### Gradient Boosting Classifier

The best Gradient Boosting Classifer achived an average precision score of 54% which is the highest of all models tested but similar to the best Random Forest Classifier and is still somewhat low.

In [ ]:
param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__n_estimators": [50, 100, 150, 200, 250, 300],
    "model__learning_rate": np.arange(0.01, 0.02, 0.02),
    "model__loss": ["log_loss", "exponential"],
    "model__criterion": ["friedman_mse", "squared_error"],
    "model__max_features": ["sqrt", "log2"]
}

best_model, best_params, best_score = tune_pipeline_hyper_parameters(
    gb_pipeline, 
    param_dist, 
    20, 
    cv, 
    "average_precision", 
    X, y
)

In [ ]:
print(f"Best paramters: {best_params}")
print(f"Best score: {best_score}")

## Final Evaluation

The best model so far seems to be the tuned Gradient Boosting Classifier. Bellow the model is evaluated in the test and achived a f1 score of 70%, average precision score of 50% and 99% roc auc score. 

In [ ]:
# Load test dataset
transactions_data_test = pd.read_csv("data/test.csv")

# Encode categorical feature
transactions_data_test = pd.get_dummies(transactions_data_test, columns=["merchant_category"], dtype="int")

In [ ]:
X = transactions_data_test.drop(["is_fraud"], axis=1)
y = transactions_data_test["is_fraud"]

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

evaluate_pipeline(best_model, X_test, y_test)

In [ ]:
plot_roc_curve_from_pipeline(best_model, X_test, y_test, plot_chance_level=True)

As seen from the confusion matrix bellow, the model corectly classifed 6 smaples from the test set as fraudulent and 389 samples as legit transctions. What is worrisome is that the false positves count and false negatives count are quite similar which could be as a result of the smaller dataset in terms of records and features. A threshold adjustment could fix this issue. 

In [ ]:
plot_confusion_matrix_from_pipeline(best_model, X_test, y_test)

## Tuning Classifiction Threshold

The optimal classifiction threshold is 47.5%. This means that if a transction has greater than 47.5% probability of being fraudulent will be classified as fraud. The thresholded Gradient Boosting Classifier achived a f1 score of 80%, an average precision score of 64% and a roc auc score of 98%. 

In [ ]:
thresholded_gb_clf = tune_classification_threshold(best_model, X_train, y_train, "average_precision", cv, )

In [ ]:
evaluate_pipeline(thresholded_gb_clf, X_test, y_test)

In [ ]:
plot_roc_curve_from_pipeline(thresholded_gb_clf, X_test, y_test, plot_chance_level=True)

As seen from the confusion matrix bellow the thresholded classifer makes less false positive classifications. The false negatives are still the same which shows that it is probably from the even smaller now after the split test dataset. 

In [ ]:
plot_confusion_matrix_from_pipeline(thresholded_gb_clf, X_test, y_test)